# Notebook 05_A SHAP Intervention Factors Ablation

Compare two intervention evidence approaches:

- **Top 10 Positive SHAP factors**
- **All Positive SHAP factors**

Comparison:
- positive SHAP evidence retained
- intervention categories represented
- intervention categories lost
- change in highest ranked intervention

In [1]:
from pathlib import Path
import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

SHAP_FILE = PROJECT_DIR / "results" / "explainability" / "at_risk_all_24_shap_factors.csv"
OUTPUT_DIR = PROJECT_DIR / "results" / "ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SHAP_FILE)
df = df[df["shap_value"] > 0].copy()

print("Predicted at risk students:", df["case_id"].nunique())


Predicted at risk students: 225


In [2]:
FEATURE_TO_INTERVENTION = {
    "absence_count": "Attendance support",
    "attendance_rate": "Attendance support",

    "previous_academic_standing": "Counselling or advisor referral",
    "previous_withdrawn_course_count": "Counselling or advisor referral",
    "programme_or_school": "Counselling or advisor referral",
    "year_level": "Counselling or advisor referral",

    "accessed_course_count": "Moodle engagement support",
    "active_days_numeric": "Moodle engagement support",
    "active_days_rate": "Moodle engagement support",
    "course_access_rate": "Moodle engagement support",
    "largest_inactivity_days": "Moodle engagement support",
    "learning_material_events_ordinal": "Moodle engagement support",
    "total_course_event_clicks": "Moodle engagement support",
    "zero_activity_days": "Moodle engagement support",

    "assessment_interaction_events_ordinal": "Tutoring and academic support",
    "credits_earned_ratio": "Tutoring and academic support",
    "previous_cgpa_ordinal": "Tutoring and academic support",
    "previous_failed_course_count": "Tutoring and academic support",
    "previous_repeated_course_count": "Tutoring and academic support",
    "previous_term_gpa_ordinal": "Tutoring and academic support",

    "enrolled_course_count": "Workload adjustment",
    "registered_course_count": "Workload adjustment",
    "registered_credits": "Workload adjustment",
    "repeated_course_count_current": "Workload adjustment",
}

df["intervention"] = df["feature"].map(FEATURE_TO_INTERVENTION)
assert df["intervention"].notna().all()

all_positive = df.copy()

top10 = (
    df.sort_values(["case_id", "shap_value"], ascending=[True, False])
      .groupby("case_id", group_keys=False)
      .head(10)
      .copy()
)


In [3]:
def summarise(selected, name):
    total_shap = df.groupby("case_id")["shap_value"].sum()
    selected_shap = selected.groupby("case_id")["shap_value"].sum()

    category_scores = (
        selected.groupby(["case_id", "intervention"])["shap_value"]
        .sum()
        .reset_index()
    )

    categories = category_scores.groupby("case_id")["intervention"].nunique()
    rank1 = (
        category_scores.sort_values(["case_id", "shap_value"], ascending=[True, False])
        .groupby("case_id")
        .first()["intervention"]
    )

    return pd.DataFrame({
        "case_id": total_shap.index,
        "Approach": name,
        "Coverage %": (selected_shap / total_shap * 100).reindex(total_shap.index).fillna(0).values,
        "Categories": categories.reindex(total_shap.index).fillna(0).values,
        "Rank 1": rank1.reindex(total_shap.index).values,
    })

top10_summary = summarise(top10, "Top 10 Positive")
all_summary = summarise(all_positive, "All Positive")

summary = pd.concat([top10_summary, all_summary], ignore_index=True)

display(
    summary.groupby("Approach")
    .agg(
        Mean_Coverage=("Coverage %", "mean"),
        Mean_Categories=("Categories", "mean"),
    )
    .round(2)
)


,Mean_Coverage,Mean_Categories
Approach,,
All Positive,100.00,4.43
Top 10 Positive,98.46,3.97


In [4]:
comparison = top10_summary.merge(
    all_summary,
    on="case_id",
    suffixes=("_Top10", "_All"),
)

comparison["Categories Lost"] = comparison["Categories_All"] - comparison["Categories_Top10"]
comparison["Rank 1 Changed"] = comparison["Rank 1_Top10"] != comparison["Rank 1_All"]

final_summary = pd.DataFrame({
    "Measure": [
        "Mean positive SHAP evidence retained by Top 10 (%)",
        "Students losing at least one intervention category",
        "Total intervention categories lost",
        "Students with changed Rank 1 intervention",
    ],
    "Value": [
        comparison["Coverage %_Top10"].mean(),
        (comparison["Categories Lost"] > 0).sum(),
        comparison["Categories Lost"].sum(),
        comparison["Rank 1 Changed"].sum(),
    ],
})

display(final_summary.round(2))

final_summary.to_csv(OUTPUT_DIR / "shap_intervention_ablation_summary.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "shap_intervention_ablation_cases.csv", index=False)


,Measure,Value
0,Mean positive SHAP evidence retained by Top 10...,98.46
1,Students losing at least one intervention cate...,88.00
2,Total intervention categories lost,102.00
3,Students with changed Rank 1 intervention,0.00
